# **HANDS - ON: INTEGRACIÓN ENTRE LLAMA Y WHATSAPP**

Una vez vista la masterclass ***Integración entre Llama y WhatsApp***, se proporciona el siguiente ***colab*** para que **el alumnado pueda practicar** haciendo uso del lenguaje de programación ***python*** y el modelo de ***llama 3.1*** de Meta.

## **LÓGICA DE NEGOCIO Y MODELO**

### **CONFIGURACIÓN DEL ENTORNO**

#### **COLAB SECRETS**

Para no exponer tu API key directamente en el código, Colab ofrece un panel de Secrets (ícono de llave en la barra lateral izquierda) donde puedes guardarla de forma segura, añadiendo un nombre asociado a la API key para guardarla dentro de una variable y usarla dentro del notebook.

In [1]:
# -----------------------------------------------------------------------------
# CREDENCIALES DE LA INTEGRACIÓN
# Esta celda recupera desde Colab Secrets los datos necesarios para comunicarse
# con WhatsApp Cloud API. La idea es no escribir credenciales sensibles
# directamente en el notebook.
# VERIFY_TOKEN se utiliza durante la verificación inicial del webhook de Meta.
# WHATSAPP_TOKEN autoriza las peticiones hacia la API de WhatsApp.
# PHONE_ID identifica el número de teléfono configurado en WhatsApp Cloud API.
# WABA_ID identifica la cuenta de WhatsApp Business asociada.
# -----------------------------------------------------------------------------

# Leer credenciales desde Colab Secrets

from google.colab import userdata

VERIFY_TOKEN = "mi_token"
WHATSAPP_TOKEN = userdata.get('WHATSAPP_ACCESS_TOKEN')
PHONE_ID = userdata.get('PHONE_NUMBER_ID')
WABA_ID = userdata.get('WABA_TOKEN_ID')

print("Credenciales cargadas correctamente.")

Credenciales cargadas correctamente.


#### **INSTALAR Y ARRANCAR OLLAMA EN COLAB**

Este módulo corre en Colab, así que Ollama no viene preinstalado en la máquina virtual: se instala y se arranca en segundo plano al inicio de cada sesión.

In [2]:
# -----------------------------------------------------------------------------
# INSTALACIÓN DE OLLAMA
# Ollama será el motor local que ejecutará el modelo Llama dentro del entorno
# de Colab. Primero se instala zstd porque es una dependencia que Ollama
# necesita para completar su instalación.
# -----------------------------------------------------------------------------

# Instalar Ollama (requiere zstd, que Colab no trae por defecto)

!apt-get update -qq
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 140 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 1s (1,141 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 126952 files and directories currently installed.)
Preparing to unpack ...

In [3]:
# -----------------------------------------------------------------------------
# ARRANQUE DE OLLAMA Y DESCARGA DEL MODELO
# Se levanta el servidor de Ollama en segundo plano para que otras partes del
# notebook puedan comunicarse con él. Después se instala el cliente de Python
# y se descarga llama3.1:8b, que será el modelo usado por el agente.
# -----------------------------------------------------------------------------

# Iniciar el servidor de Ollama en segundo plano

import subprocess
import time

# Se ejecuta `ollama serve` como proceso separado para mantener disponible el modelo.
proceso_ollama = subprocess.Popen(["ollama", "serve"])
time.sleep(5)  # dar tiempo a que el servidor levante

!pip install ollama --quiet
import ollama

# Descarga el modelo que posteriormente será registrado también en Llama Stack.
!ollama pull llama3.1:8b
print("Ollama listo.")


Ollama listo.


#### **INSTALAR Y LEVANTAR LLAMA STACK**

In [4]:
# -----------------------------------------------------------------------------
# DEPENDENCIAS DE LLAMA STACK
# Llama Stack organiza componentes del agente, como conversaciones, respuestas
# y herramientas. También se instala una versión compatible de MCP.
# -----------------------------------------------------------------------------

# Instalar dependencias

!pip install "llama-stack[starter]" llama-stack-client --quiet
!pip install "mcp<2.0" --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 23.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.3/89.3 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.3/375.3 kB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 952.8/952.8 kB 53.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 569.2/569.2 kB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.7/319.7 kB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.7/365.7 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 83.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 89.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.0/80.0 kB 8.1 MB/s eta 0:00:00
  

In [5]:
# -----------------------------------------------------------------------------
# CONFIGURACIÓN Y ARRANQUE DE LLAMA STACK
# Esta celda conecta Llama Stack con el modelo que está sirviendo Ollama.
# Se toma la configuración starter, se registra llama3.1:8b y se guarda una
# configuración propia. Luego se levanta Llama Stack en el puerto 8321.
# El cliente creado al final será el que usemos para enviar mensajes al modelo.
# -----------------------------------------------------------------------------

# Instalar Llama Stack (servidor + cliente) y levantarlo con Ollama registrado

import subprocess
import time
import os
import yaml
import llama_stack

# Llama Stack necesita saber en qué dirección está escuchando Ollama.
os.environ["OLLAMA_URL"] = "http://localhost:11434/v1"

config_path = os.path.join(os.path.dirname(llama_stack.__file__), "distributions", "starter", "config.yaml")
with open(config_path) as f:
    config = yaml.safe_load(f)

# Se agrega el modelo de Ollama al catálogo de modelos disponibles en Llama Stack.
config["registered_resources"]["models"].append({
    "model_id": "llama3.1:8b",
    "provider_id": "ollama",
    "provider_model_id": "llama3.1:8b",
    "model_type": "llm",
})

mi_config_path = "/content/mi_config_llama_stack.yaml"
with open(mi_config_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

# Se levanta Llama Stack como un servicio separado para que el cliente pueda consultarlo.
proceso_llama_stack = subprocess.Popen(
    ["llama", "stack", "run", mi_config_path, "--port", "8321"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
time.sleep(20)  # tarda mas en levantar que Ollama

print("Proceso vivo:", proceso_llama_stack.poll() is None)
if proceso_llama_stack.poll() is not None:
    print("--- Salida del proceso ---")
    print(proceso_llama_stack.stdout.read())
else:
    from llama_stack_client import LlamaStackClient
    client = LlamaStackClient(base_url="http://localhost:8321", timeout=300)
    print("Llama Stack listo.")

Proceso vivo: True
Llama Stack listo.


### **CONSTRUCCIÓN DE LA LÓGICA DEL ASISTENTE**

In [6]:
# -----------------------------------------------------------------------------
# SERVIDOR DEL WEBHOOK
# FastAPI recibe las notificaciones HTTP que Meta envía cuando llega un mensaje.
# mensajes_recibidos permite conservar evidencia de los payloads recibidos.
# ids_procesados evita procesar dos veces el mismo mensaje si Meta lo reintenta.
# -----------------------------------------------------------------------------

# Configurar el servidor del webhook

from fastapi import FastAPI, Request, BackgroundTasks
import requests
import json

app = FastAPI()

mensajes_recibidos = []
ids_procesados = set()

In [7]:
# -----------------------------------------------------------------------------
# NORMALIZACIÓN DEL NÚMERO TELEFÓNICO
# Esta función corrige un formato específico de números mexicanos que puede
# llegar desde WhatsApp antes de usarlo como identificador interno.
# -----------------------------------------------------------------------------

# Definir función de normalización de número

def normalizar_numero_mx(numero):
    if numero.startswith("521") and len(numero) == 13:
        return "52" + numero[3:]
    return numero

#### **CONSTRUIR LA MEMORIA DE CONVERSACIÓN**

Se define una estructura en memoria que asocia cada número de teléfono con su `conversation_id` de **Llama Stack** — sin esto, Llama trataría cada mensaje como si fuera el primero de la conversación.

In [8]:
# -----------------------------------------------------------------------------
# MEMORIA DE CONVERSACIÓN POR USUARIO
# Como explicó el sensei, cada número de teléfono se asocia con una conversación
# independiente en Llama Stack. Así el agente puede mantener el contexto de
# cada usuario entre mensajes sin mezclar conversaciones.
# -----------------------------------------------------------------------------

# Una conversacion de Llama Stack por numero de telefono

conversaciones_por_numero = {}

# Recupera la conversación existente o crea una nueva si es el primer mensaje de ese número.
def obtener_conversacion(numero):
    if numero not in conversaciones_por_numero:
        conversacion = client.conversations.create()
        conversaciones_por_numero[numero] = conversacion.id
    return conversaciones_por_numero[numero]

#### **DEFINIR LA HERRAMIENTA: CONSULTAR DISPONIBILIDAD Y PREGUNTAS FRECUENTES**

Hasta el Tema anterior, Llama solo generaba texto a partir de texto — si un usuario preguntaba por un horario, el modelo podía inventar una respuesta que sonara razonable, sin ninguna garantía de que fuera real. **Function calling** resuelve esto: se le describe a Llama una función real de tu código (nombre, qué hace, qué parámetros necesita), y el modelo, cuando lo considera necesario, pide que se ejecute esa función en vez de responder directamente.

Aquí se definen dos herramientas: `consultar_disponibilidad`, que revisa horarios reales del taller, y `buscar_pregunta_frecuente`, que busca la respuesta a dudas comunes (horarios, garantía, formas de pago) en una base de preguntas frecuentes real — una alternativa más ligera a montar un pipeline de RAG completo (embeddings + recuperación) aparte: en vez de eso, la búsqueda se expone como una función más que Llama puede pedir ejecutar, bajo el mismo mecanismo de function calling.

In [9]:
# -----------------------------------------------------------------------------
# TOOL 1: CONSULTAR DISPONIBILIDAD
# Aquí se define tanto la función real de Python como el contrato que Llama ve.
# La función consulta datos reales del ejemplo del taller; la descripción de la
# herramienta le indica al modelo cuándo usarla y qué parámetro puede enviar.
# Esto ejemplifica function calling: Llama solicita una acción y el código la ejecuta.
# -----------------------------------------------------------------------------

# Definir la funcion real y su descripcion para Llama: consultar disponibilidad

import re
from datetime import datetime

# Base de datos de ejemplo que representa los horarios reales que la tool puede consultar.
DISPONIBILIDAD = {
    "2026-10-06": ["09:00", "11:00", "15:00"],
    "2026-10-07": ["10:00", "13:00"],
}

MESES = {
    "enero": 1, "febrero": 2, "marzo": 3, "abril": 4, "mayo": 5, "junio": 6,
    "julio": 7, "agosto": 8, "septiembre": 9, "setiembre": 9, "octubre": 10,
    "noviembre": 11, "diciembre": 12,
}

# Convierte distintas formas de escribir una fecha a un formato uniforme YYYY-MM-DD.
def normalizar_fecha(fecha_texto):
    fecha_texto = fecha_texto.strip().lower()

    if re.fullmatch(r"\d{4}-\d{2}-\d{2}", fecha_texto):
        _, mes, dia = fecha_texto.split("-")
        return f"{datetime.now().year:04d}-{mes}-{dia}"

    match = re.search(r"(\d{1,2})\s+de\s+([a-zñ]+)(?:\s+de\s+(\d{4}))?", fecha_texto)
    if match:
        dia, mes_texto, anio_texto = match.groups()
        mes = MESES.get(mes_texto)
        if mes:
            anio = int(anio_texto) if anio_texto else datetime.now().year
            return f"{anio:04d}-{mes:02d}-{int(dia):02d}"

    return fecha_texto

# Implementación real de la acción que el modelo puede solicitar mediante function calling.
def consultar_disponibilidad(fecha=None):
    if not fecha:
        dias = ", ".join(DISPONIBILIDAD.keys())
        return f"Los dias con horarios disponibles son: {dias}. Dime cual te interesa para darte los horarios exactos."

    fecha = normalizar_fecha(fecha)
    horarios = DISPONIBILIDAD.get(fecha, [])
    if not horarios:
        return f"No hay horarios disponibles para el {fecha}."
    return f"Horarios disponibles para el {fecha}: {', '.join(horarios)}."

# Contrato descriptivo que recibe Llama: nombre, propósito y esquema de parámetros.
HERRAMIENTA_DISPONIBILIDAD = {
    "type": "function",
    "name": "consultar_disponibilidad",
    "description": "Consulta los horarios disponibles para llevar el auto al taller. Si el usuario da una fecha especifica, se le devuelven los horarios de ese dia; si pregunta de forma general (sin fecha), se le devuelve la lista de dias con disponibilidad.",
    "parameters": {
        "type": "object",
        "properties": {
            "fecha": {
                "type": "string",
                "description": "Fecha a consultar, en formato YYYY-MM-DD. Si el usuario no menciona el año, asume que es el año actual. Omite este parametro si el usuario no dio una fecha especifica."
            }
        },
        "required": []
    }
}

In [10]:
# -----------------------------------------------------------------------------
# TOOL 2: BUSCAR PREGUNTAS FRECUENTES
# Esta herramienta consulta una base sencilla de respuestas del taller.
# Primero se normalizan acentos para hacer más tolerante la búsqueda.
# HERRAMIENTAS contiene los contratos visibles para el modelo.
# FUNCIONES_DISPONIBLES conecta el nombre solicitado por Llama con la función
# real de Python que debe ejecutarse.
# -----------------------------------------------------------------------------

# Definir la funcion real y su descripcion para Llama: buscar pregunta frecuente

import unicodedata

# Facilita que 'cotización' y 'cotizacion', por ejemplo, se comparen de la misma forma.
def quitar_acentos(texto):
    return ''.join(c for c in unicodedata.normalize('NFD', texto) if unicodedata.category(c) != 'Mn')

# Base de conocimiento sencilla utilizada por la herramienta FAQ.
PREGUNTAS_FRECUENTES = {
    "horario": "El taller abre de lunes a sabado, de 8:00 a 18:00.",
    "garantia": "Todas las reparaciones tienen 90 dias de garantia en mano de obra.",   # raiz, no palabra completa
    "pago": "Aceptamos efectivo, tarjeta y transferencia.",
    "cotizacion": "La cotizacion es gratuita, solo se cobra si decides hacer la reparacion.",  # raiz
}

# Busca una raíz temática dentro de la pregunta y devuelve la respuesta almacenada.
def buscar_pregunta_frecuente(pregunta):
    pregunta_norm = quitar_acentos(pregunta.lower())
    for raiz, respuesta in PREGUNTAS_FRECUENTES.items():
        if raiz in pregunta_norm:
            return respuesta
    return "No tenemos una respuesta guardada para esa pregunta; un miembro del equipo puede confirmarte directamente."

# Esquema que explica al modelo qué hace la herramienta y qué argumento necesita.
HERRAMIENTA_FAQ = {
    "type": "function",
    "name": "buscar_pregunta_frecuente",
    "description": "Busca la respuesta a una pregunta frecuente del taller (horarios, garantia, formas de pago, cotizaciones) en una base de preguntas frecuentes real.",
    "parameters": {
        "type": "object",
        "properties": {
            "pregunta": {
                "type": "string",
                "description": "La pregunta del usuario, en texto libre."
            }
        },
        "required": ["pregunta"]
    }
}

# Catálogo de tools que Llama podrá elegir durante la generación.
HERRAMIENTAS = [HERRAMIENTA_DISPONIBILIDAD, HERRAMIENTA_FAQ]
# Mapeo del nombre solicitado por el modelo hacia la función ejecutable del backend.
FUNCIONES_DISPONIBLES = {
    "consultar_disponibilidad": consultar_disponibilidad,
    "buscar_pregunta_frecuente": buscar_pregunta_frecuente,
}

#### **GENERAR LA RESPUESTA CON HISTORIAL Y FUNCTION CALLING**

Se reenvía el historial reciente a Llama junto con las dos herramientas disponibles. Si el modelo pide ejecutar alguna (`respuesta.output` trae uno o más elementos con `type == "function_call"`), el código la ejecuta de verdad con los argumentos que Llama propuso, le devuelve el resultado en una segunda llamada, y solo entonces se genera la respuesta final en lenguaje natural. Las reglas fijas (urgencia y escalamiento) se resuelven antes de llegar aquí — ver la sección de endpoints.

In [11]:
# -----------------------------------------------------------------------------
# GENERACIÓN DE RESPUESTA + MEMORIA + FUNCTION CALLING
# El SYSTEM_PROMPT delimita el comportamiento del agente y evita que invente
# acciones que el sistema no puede realizar, como confirmar una cita.
# generar_respuesta envía el texto del usuario a Llama dentro de su conversación.
# Si el modelo devuelve un function_call, el backend ejecuta la herramienta real,
# devuelve su resultado al modelo y solicita una segunda respuesta ya sustentada
# en ese resultado. Este es el ciclo de function calling explicado en la clase.
# -----------------------------------------------------------------------------

# Definir la funcion de generacion de respuesta con memoria y function calling

# Instrucciones permanentes del agente: alcance, restricciones y forma de responder.
SYSTEM_PROMPT = (
    "Eres el asistente de un taller mecanico. "
    "Responde de forma breve (2-3 lineas). "
    "Solo usa una herramienta cuando el usuario pregunte algo especifico que la herramienta pueda responder "
    "(disponibilidad de una fecha, o una pregunta frecuente concreta). No uses ninguna herramienta para "
    "confirmaciones o respuestas cortas del usuario (como 'si, por favor' o una hora). "
    "No puedes agendar, confirmar ni programar citas de ninguna forma — solo puedes consultar disponibilidad. "
    "Si el usuario quiere agendar una cita, dile que un miembro del equipo se pondra en contacto para confirmarla. "
    "Si una herramienta no encuentra una respuesta, dilo honestamente en vez de inventar un dato. "
    "Si una herramienta SI regresa un resultado real, siempre comunicalo claramente al usuario; nunca lo ignores ni te niegues a compartirlo. "
    "Responde siempre de forma directa con la informacion final, basandote unicamente en el resultado real de las "
    "herramientas; nunca inventes datos que no vengan de una herramienta. "
    "Nunca describas tu proceso, ni menciones nombres de funciones, parametros o claves de un resultado."
)

# Punto central de interacción con Llama para cada mensaje que no resolvieron las reglas fijas.
def generar_respuesta(numero, texto_usuario):
    conversation_id = obtener_conversacion(numero)
# Con el resultado de la herramienta se vuelve a consultar Llama para redactar la respuesta final.
    respuesta = client.responses.create(
        model="ollama/llama3.1:8b",
        input=texto_usuario,
        conversation=conversation_id,
        instructions=SYSTEM_PROMPT,
        tools=HERRAMIENTAS,
    )

# Se filtran las salidas estructuradas de tipo function_call.
    llamadas_funcion = [item for item in respuesta.output if item.type == "function_call"]
# Si Llama pidió una acción, el backend debe ejecutarla; el modelo no ejecuta Python por sí mismo.
    if llamadas_funcion:
        resultados = []
        for llamada in llamadas_funcion:
# Localiza la función real correspondiente al nombre de herramienta elegido por Llama.
            funcion = FUNCIONES_DISPONIBLES[llamada.name]
# Convierte los argumentos estructurados generados por el modelo a un diccionario de Python.
            argumentos = json.loads(llamada.arguments)
# Aquí ocurre la acción real en el backend con los argumentos solicitados.
            resultado = funcion(**argumentos)
            resultados.append({
                "type": "function_call_output",
                "call_id": llamada.call_id,
                "output": resultado,
            })
        respuesta = client.responses.create(
            model="ollama/llama3.1:8b",
            input=resultados,
            conversation=conversation_id,
            instructions=SYSTEM_PROMPT,
            tools=HERRAMIENTAS,
        )

# Devuelve únicamente el texto final que posteriormente será enviado al usuario por WhatsApp.
    return respuesta.output_text

## **DESPLIEGUE DEL SERVIDOR**

### **DEFINIR LOS ENDPOINTS DEL WEBHOOK**

Se definen los dos endpoints que WhatsApp necesita: uno para la verificación inicial del webhook, y otro para recibir cada mensaje entrante. La recepción delega el trabajo pesado a `BackgroundTasks`, para responderle a Meta de inmediato sin esperar a que Llama genere la respuesta — evitando así que Meta reintente la entrega y se dupliquen las respuestas.

In [12]:
# -----------------------------------------------------------------------------
# ENDPOINT GET: VERIFICACIÓN DEL WEBHOOK
# Meta llama este endpoint durante la configuración inicial del webhook.
# Si el verify token coincide, el servidor devuelve el challenge que Meta espera.
# -----------------------------------------------------------------------------

# Definir endpoint de verificacion de credenciales

@app.get("/webhook")
async def verificar_webhook(request: Request):
    params = request.query_params
# Meta solo valida el webhook si el token recibido coincide con el definido en el notebook.
    if params.get("hub.verify_token") == VERIFY_TOKEN:
        return int(params.get("hub.challenge"))
    return {"error": "token invalido"}

In [13]:
# -----------------------------------------------------------------------------
# ENDPOINT POST: RECEPCIÓN DE MENSAJES
# Este es el punto de entrada de los mensajes reales de WhatsApp.
# Se extraen del payload el ID del mensaje, el número del usuario y el texto.
# El procesamiento se envía a BackgroundTasks para responder después de devolver
# rápidamente un 200 OK a Meta. También se evita reprocesar mensajes repetidos.
# -----------------------------------------------------------------------------

# Definir endpoint de recepcion de mensajes

@app.post("/webhook")
async def recibir_mensaje(request: Request, background_tasks: BackgroundTasks):
# Convierte el cuerpo JSON enviado por Meta en una estructura de Python.
    body = await request.json()
    mensajes_recibidos.append(body)
    try:
# Navega por el payload de Meta hasta obtener el primer mensaje entrante.
        mensaje = body["entry"][0]["changes"][0]["value"]["messages"][0]
        wamid = mensaje["id"]
# WhatsApp puede reintentar eventos; este control evita respuestas duplicadas.
        if wamid in ids_procesados:
            return {"status": "ok"}  # Meta reenvio el mismo mensaje; ya se proceso, se ignora
        ids_procesados.add(wamid)
        numero = normalizar_numero_mx(mensaje["from"])
        texto = mensaje["text"]["body"]
# El procesamiento pesado se ejecuta después, sin retrasar la confirmación HTTP a Meta.
        background_tasks.add_task(procesar_y_responder, numero, texto)
    except (KeyError, IndexError):
        pass
    return {"status": "ok"}

In [14]:
# -----------------------------------------------------------------------------
# ENVÍO DE LA RESPUESTA A WHATSAPP
# Esta función construye la petición HTTP hacia Graph API.
# El payload indica el destinatario y el texto que verá el usuario.
# Se conserva el status y la respuesta de Meta para facilitar la depuración.
# -----------------------------------------------------------------------------

# Definir funcion de envio de respuesta

respuestas_enviadas = []

def enviar_respuesta(numero, texto):
# Endpoint de Graph API utilizado para enviar mensajes con el Phone Number ID configurado.
    url = f"https://graph.facebook.com/v25.0/{PHONE_ID}/messages"
    headers = {"Authorization": f"Bearer {WHATSAPP_TOKEN}", "Content-Type": "application/json"}
# Estructura requerida por WhatsApp Cloud API para un mensaje de texto.
    payload = {
        "messaging_product": "whatsapp",
        "to": numero,
        "type": "text",
        "text": {"body": texto}
    }

# Envía la respuesta final al usuario a través de la API de Meta.
    r = requests.post(url, headers=headers, json=payload)
    respuestas_enviadas.append({"numero": numero, "status": r.status_code, "texto": r.text})

In [15]:
# -----------------------------------------------------------------------------
# BLOQUEO POR USUARIO
# Un Lock por número evita que dos mensajes del mismo usuario se procesen al
# mismo tiempo y alteren el orden lógico de su conversación.
# -----------------------------------------------------------------------------

# Un lock por numero de telefono, para serializar mensajes del mismo usuario

import threading

locks_por_numero = {}  # { numero: threading.Lock() }

# Crea el Lock la primera vez y lo reutiliza en los siguientes mensajes del mismo número.
def obtener_lock(numero):
    if numero not in locks_por_numero:
        locks_por_numero[numero] = threading.Lock()
    return locks_por_numero[numero]

In [16]:
# -----------------------------------------------------------------------------
# ORQUESTACIÓN DEL MENSAJE
# Esta función decide qué camino sigue cada mensaje.
# Algunas situaciones críticas se resuelven con reglas fijas: urgencia,
# escalamiento a humano y saludos. El resto se delega a Llama.
# También se clasifican errores de Llama Stack, Ollama y WhatsApp para registrar
# la causa y, cuando es posible, avisar al usuario de forma controlada.
# -----------------------------------------------------------------------------

# Definir funcion de procesamiento: reglas fijas de urgencia, escalamiento y saludo, o respuesta de Llama

import requests
from llama_stack_client import APITimeoutError, APIConnectionError, APIStatusError

# Palabras o frases que activan una respuesta crítica definida por regla.
FRASES_URGENCIA = ["me quede varado", "no arranca", "choque", "tuve un accidente", "emergencia"]
FRASES_ESCALAMIENTO = ["hablar con alguien", "hablar con una persona", "asistente humano"]
FRASES_SALUDO = ["hola", "buenas", "buenos dias", "buenos días", "buenas tardes", "buenas noches", "que tal", "buen dia", "buen día"]

def es_solo_saludo(texto_low):
    texto_limpio = texto_low.strip(" ,.!¡¿?")
    return texto_limpio in FRASES_SALUDO

def enviar_aviso_de_error(numero):
    try:
        enviar_respuesta(numero, "Estamos teniendo un problema tecnico. Intenta de nuevo en un momento.")
    except Exception:
        pass  # si ni el aviso de error se pudo mandar, no hay mas que intentar aqui

errores_procesamiento = []

# Orquesta el flujo completo para un mensaje recibido desde WhatsApp.
def procesar_y_responder(numero, texto):
# Garantiza que los mensajes de un mismo usuario se atiendan secuencialmente.
    with obtener_lock(numero):
        try:
            texto_low = texto.lower()
# Regla fija 1: una urgencia se atiende sin delegar la decisión al modelo.
            if any(frase in texto_low for frase in FRASES_URGENCIA):
                enviar_respuesta(numero, "Eso suena urgente: comunicate directamente a nuestra linea de asistencia en carretera.")
# Regla fija 2: una solicitud explícita de humano activa el escalamiento.
            elif any(frase in texto_low for frase in FRASES_ESCALAMIENTO):
                enviar_respuesta(numero, "Claro, en un momento un miembro del equipo te contacta directamente.")
# Regla fija 3: un saludo simple recibe una respuesta directa.
            elif es_solo_saludo(texto_low):
                enviar_respuesta(numero, "Hola, soy el asistente del taller mecanico. ¿En que puedo ayudarte?")
            else:
# Las consultas restantes pasan a Llama, que podrá responder o usar una tool.
                respuesta = generar_respuesta(numero, texto)
                enviar_respuesta(numero, respuesta)

# Maneja una generación que excedió el tiempo máximo de espera.
        except APITimeoutError as e:
            errores_procesamiento.append({
                "numero": numero, "texto": texto, "causa": "llama_timeout",
                "detalle": "Llama Stack conecto pero la generacion tardo demasiado.",
                "error": repr(e),
            })
            enviar_aviso_de_error(numero)

# Distingue si la falla de conexión proviene de Llama Stack, Ollama u otra causa.
        except APIConnectionError as e:
            llama_stack_vivo = proceso_llama_stack.poll() is None
            ollama_vivo = proceso_ollama.poll() is None
            if not llama_stack_vivo:
                causa, detalle = "llama_stack_caido", f"El proceso de Llama Stack ya no esta corriendo (Ollama vivo: {ollama_vivo})."
            elif not ollama_vivo:
                causa, detalle = "ollama_caido", "El proceso de Ollama ya no esta corriendo."
            else:
                causa, detalle = "conexion_desconocida", "No se pudo conectar a Llama Stack aunque ambos procesos parecen vivos."
            errores_procesamiento.append({"numero": numero, "texto": texto, "causa": causa, "detalle": detalle, "error": repr(e)})
            enviar_aviso_de_error(numero)

# Registra errores HTTP devueltos por Llama Stack.
        except APIStatusError as e:
            errores_procesamiento.append({
                "numero": numero, "texto": texto, "causa": "llama_stack_error_http",
                "detalle": f"Llama Stack respondio con error HTTP {e.status_code}.",
                "error": repr(e),
            })
            enviar_aviso_de_error(numero)

# Captura el caso en que el modelo solicite una herramienta no registrada.
        except KeyError as e:
            errores_procesamiento.append({
                "numero": numero, "texto": texto, "causa": "herramienta_no_reconocida",
                "detalle": f"El modelo pidio ejecutar una herramienta que no existe: {e}.",
                "error": repr(e),
            })
            enviar_aviso_de_error(numero)

# Captura problemas al comunicarse con la API de WhatsApp.
        except requests.exceptions.RequestException as e:
            # Este es el unico que NO intenta avisar al usuario: si fallo mandar el mensaje
            # a WhatsApp, intentar mandar el aviso de error fallaria por la misma razon.
            errores_procesamiento.append({
                "numero": numero, "texto": texto, "causa": "whatsapp_api_error",
                "detalle": "Fallo el envio a la API de WhatsApp (graph.facebook.com).",
                "error": repr(e),
            })

        except Exception as e:
            errores_procesamiento.append({
                "numero": numero, "texto": texto, "causa": "desconocido",
                "detalle": "Error no clasificado.",
                "error": repr(e),
            })
            enviar_aviso_de_error(numero)

### **EXPONER EL SERVIDOR CON NGROK**

Se abre un túnel público con ngrok y se levanta el servidor en un hilo aparte, para que Meta pueda entregarle los mensajes reales al webhook. La URL impresa (+ `/webhook`) es la que se configura como *Callback URL* en Meta for Developers (WhatsApp > Configuración de la API > Configurar webhooks).

In [18]:
# -----------------------------------------------------------------------------
# EXPOSICIÓN PÚBLICA DEL WEBHOOK CON NGROK
# Colab no tiene una URL pública estable para recibir eventos desde Meta.
# ngrok crea un túnel temporal hacia el puerto 8000 donde corre FastAPI.
# La URL impresa es la que se configura como Callback URL en Meta for Developers.
# -----------------------------------------------------------------------------

# Instalar e iniciar tunel publico con ngrok

!pip install pyngrok --quiet

from pyngrok import ngrok
import uvicorn
import threading

ngrok.set_auth_token(userdata.get('NGROK_AUTHTOKEN'))
# Expone públicamente el mismo puerto donde se levantará FastAPI.
tunel_publico = ngrok.connect(8000)
print("URL a configurar como Callback URL en el dashboard de Meta:")
print(tunel_publico.public_url + "/webhook")

# Uvicorn ejecuta la aplicación FastAPI que contiene los endpoints /webhook.
def correr_servidor():
    uvicorn.run(app, host="0.0.0.0", port=8000)

# Se usa un hilo para que el servidor quede activo sin bloquear la ejecución del notebook.
hilo = threading.Thread(target=correr_servidor, daemon=True)
hilo.start()

URL a configurar como Callback URL en el dashboard de Meta:
https://zombie-overpay-rectangle.ngrok-free.dev/webhook


### **PROBAR EL FLUJO COMPLETO**

Se manda un mensaje preguntando por horarios disponibles en una fecha específica, para confirmar que la respuesta viene de `consultar_disponibilidad` y no de una invención de Llama; y un segundo mensaje pidiendo hablar con una persona, para confirmar que la regla fija de escalamiento sigue funcionando.

In [ ]:
# -----------------------------------------------------------------------------
# ESCENARIO DE PRUEBA 1
# Prueba el uso de la herramienta de disponibilidad en una conversación de varios
# turnos, empezando con un saludo y terminando con una fecha concreta.
# -----------------------------------------------------------------------------

# Escenario 1: Mandar mensajes al bot por disponibilidad general

# Mensaje 1:
# Buenas tardes.

# Mensaje 2:
# Necesito llevar mi carro al taller, ¿qué días tienen espacio?

# Mensaje 3:
# Perfecto, ¿y a qué hora el 6 de octubre?

In [ ]:
# -----------------------------------------------------------------------------
# ESCENARIO DE PRUEBA 2
# Valida las reglas fijas para una urgencia y el escalamiento a una persona.
# Estas decisiones no se dejan al modelo porque son rutas críticas del flujo.
# -----------------------------------------------------------------------------

# Escenario 2: Mandar mensajes al bot por tema de urgencia

# Mensaje 1:
# Hola. Necesito ayuda.
# Acabo de chocar y mi carro no arranca.

# Mensaje 2:
# Comunícame con un asistente humano.

In [ ]:
# -----------------------------------------------------------------------------
# ESCENARIO DE PRUEBA 3
# Valida la herramienta de preguntas frecuentes y el límite del agente:
# puede consultar información, pero no puede confirmar una cita por sí mismo.
# -----------------------------------------------------------------------------

# Escenario 3: Mandar mensajes al bot por pago y cotización

# Mensaje 1:
# Buenas, ¿cuánto cuesta una cotización?

# Mensaje 2:
# ¿Y qué métodos de pago aceptan?

# Mensaje 3:
# Perfecto, entonces agéndame para el 7 de octubre a las 10.

In [ ]:
# -----------------------------------------------------------------------------
# ESCENARIO DE PRUEBA 4
# Comprueba cómo responde el agente cuando la pregunta está fuera de la
# información disponible en sus herramientas.
# -----------------------------------------------------------------------------

# Escenario 4: Mandar mensajes al bot por temas fuera de alcance

# Mensaje 1:
# Hola, ¿tienen servicio de grúa?

# Mensaje 2:
# ¿También atienden motos, o solo carros?

In [24]:
# -----------------------------------------------------------------------------
# VISUALIZACIÓN DE EVIDENCIAS
# Muestra el estado de las conversaciones, payloads recibidos, respuestas enviadas
# y errores registrados para revisar el ciclo completo de la integración.
# -----------------------------------------------------------------------------

# Visualizar resultados

# Cada impresión ayuda a revisar una parte distinta del ciclo end-to-end.
print(f'Conversaciones: {conversaciones_por_numero}\n')
print(f'Mensajes recibidos: {mensajes_recibidos}\n')
print(f'Respuestas enviadas: {respuestas_enviadas}\n')
print(f'Errores procesados: {errores_procesamiento}\n')

Conversaciones: {}

Mensajes recibidos: []

Respuestas enviadas: []

Errores procesados: []



In [25]:
# -----------------------------------------------------------------------------
# INSPECCIÓN DE LA MEMORIA EN LLAMA STACK
# Permite observar los elementos almacenados en una conversación:
# mensajes normales, llamadas a herramientas y resultados de esas llamadas.
# Es útil para verificar que el contexto y el function calling quedaron registrados.
# -----------------------------------------------------------------------------

# Verificar almacenamiento en Llama Stack

# Número usado como clave para localizar su conversación almacenada en Llama Stack.
numero_a_revisar = '529621079387'

if numero_a_revisar not in conversaciones_por_numero:
    print("No hay mensajes almacenados en estos momentos.")
else:
    conv_id = conversaciones_por_numero[numero_a_revisar]
# Recupera el historial completo de elementos de esa conversación.
    items = client.conversations.items.list(conversation_id=conv_id)

    for item in items.data:
        if item.type == "message":
            print(item.role, "->", item.content)
        elif item.type == "function_call":
            print("function_call ->", item.name, item.arguments)
        elif item.type == "function_call_output":
            print("function_call_output ->", item.output)
        else:
            print(item.type, "->", item)

No hay mensajes almacenados en estos momentos.


### **DIAGNÓSTICO Y REINICIO**

Si el bot responde con datos que no corresponden al taller (fechas, textos u otro negocio), antes de reiniciar todo el entorno revisa primero si simplemente falta volver a correr alguna celda de arriba.

In [26]:
# -----------------------------------------------------------------------------
# REINICIO DEL ALMACENAMIENTO DE LLAMA STACK
# Detiene el proceso, elimina el almacenamiento local de la distribución starter
# y vuelve a levantar Llama Stack para comenzar una prueba limpia.
# -----------------------------------------------------------------------------

# Reiniciar el almacenamiento en Llama Stack

# Se detiene el servicio antes de borrar su almacenamiento local.
proceso_llama_stack.terminate()
import time; time.sleep(3)

# Elimina el estado persistido para que las siguientes pruebas comiencen sin historial.
!rm -rf ~/.llama/distributions/starter

proceso_llama_stack = subprocess.Popen(
    ["llama", "stack", "run", mi_config_path, "--port", "8321"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
time.sleep(20)  # tarda mas en levantar que Ollama

print("Proceso vivo:", proceso_llama_stack.poll() is None)
if proceso_llama_stack.poll() is not None:
    print("--- Salida del proceso ---")
    print(proceso_llama_stack.stdout.read())
else:
    from llama_stack_client import LlamaStackClient
    client = LlamaStackClient(base_url="http://localhost:8321", timeout=300)
    print("Llama Stack listo.")

Proceso vivo: True
Llama Stack listo.


In [ ]:
# -----------------------------------------------------------------------------
# LIMPIEZA DE REGISTROS EN MEMORIA DEL NOTEBOOK
# Vacía las listas y diccionarios usados para observar pruebas anteriores.
# No modifica la lógica del agente; únicamente deja los registros locales limpios.
# -----------------------------------------------------------------------------

# Limpiar los registros en memoria, para una prueba nueva sin ruido de mensajes anteriores

mensajes_recibidos = []
respuestas_enviadas = []
errores_procesamiento = []
conversaciones_por_numero = {}

In [ ]:
# -----------------------------------------------------------------------------
# CIERRE DEL TÚNEL NGROK
# Finaliza la exposición pública del servidor cuando ya terminaron las pruebas.
# -----------------------------------------------------------------------------

# Finalizar proceso de tunelizacion

ngrok.kill()
print('Proceso de ngrok cerrado.')

Proceso de ngrok cerrado.


In [ ]:
# -----------------------------------------------------------------------------
# IDENTIFICADOR DE LA CUENTA DE WHATSAPP BUSINESS
# Esta celda deja un valor de ejemplo para el WABA_ID usado en las consultas
# posteriores de suscripción de la aplicación.
# -----------------------------------------------------------------------------

# "Configuración de la API" de Meta, normalmente arriba del Phone Number ID

WABA_ID = "XXXXXXXXXXXXXXX"

In [ ]:
# -----------------------------------------------------------------------------
# CONSULTA DE SUSCRIPCIONES DEL WABA
# Solicita a Graph API la lista de aplicaciones actualmente suscritas a la cuenta
# de WhatsApp Business para comprobar la configuración.
# -----------------------------------------------------------------------------

# 1. Ver qué apps están suscritas a este WABA

# GET consulta el estado actual; no modifica todavía la suscripción.
r = requests.get(
    f"https://graph.facebook.com/v21.0/{WABA_ID}/subscribed_apps",
    headers={"Authorization": f"Bearer {WHATSAPP_TOKEN}"}
)
print("Suscripciones actuales:", r.json())

In [ ]:
# -----------------------------------------------------------------------------
# SUSCRIPCIÓN DE LA APP AL WABA
# Envía una petición POST para suscribir la aplicación a la cuenta de WhatsApp
# Business y muestra la respuesta de Graph API.
# -----------------------------------------------------------------------------

# 2. Suscribir esta app al WABA

# POST solicita el cambio: suscribir esta aplicación al WABA.
r2 = requests.post(
    f"https://graph.facebook.com/v21.0/{WABA_ID}/subscribed_apps",
    headers={"Authorization": f"Bearer {WHATSAPP_TOKEN}"}
)
print("Resultado de suscripción:", r2.json())